In [ ]:
# from langchain.prompts import PromptTemplate, SystemMessagePromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate
# from langchain.agents import load_tools, initialize_agent, AgentType, create_react_agent, AgentExecutor
# from langchain.memory import ConversationBufferMemory
# from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [66]:
# dotenv_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
# load_dotenv(dotenv_path)

# API_KEY = os.getenv("GOOGLE_API_KEY")

In [67]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=API_KEY, temperature=0)
# memory = ConversationBufferMemory(memory_key='chat_history')
# tools = load_tools(['pubmed'], llm=llm)
# agent = initialize_agent(tools, llm, agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION, memory=memory, verbose=True)

In [68]:
# system_message_text = f"""
# Eres un agente .....
# """

In [69]:
# memory.chat_memory.add_message(SystemMessage(system_message_text))

# result = agent.invoke("hola")['output']

In [70]:
from langchain.agents import initialize_agent, Tool, load_tools
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import BaseTool
from langchain.memory import ConversationBufferMemory
import pandas as pd
import fitz

import re
import json
import os
from dotenv import load_dotenv

In [71]:
dotenv_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
load_dotenv(dotenv_path)

API_KEY = os.getenv("GOOGLE_API_KEY")

In [72]:
# Tool para leer un PDF
class LeerPDFTool(BaseTool):
    name : str = "leer_pdf"
    description : str = "Lee información de un PDF sobre un producto"

    def _run(self, file_path: str):
        doc = fitz.open(file_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    


# Tool para consultar CSVs con factores de emisión
class ConsultarCSVTool(BaseTool):
    name : str = "consultar_csv"
    description : str = "Consulta factores de emisión en archivos CSV"

    def _run(self, query: str):
        procesos = pd.read_csv("data/processed/openlca_results_recorrected_calcsetup_ELCD.csv")   #### DEFINIR BIEN LOS CSVs ####
        return {
            "procesos": procesos.to_dict(orient='records'),
        }
    
 

# Tool para calcular la huella de carbono (ejemplo simple)
class CalculadoraEmisionesTool(BaseTool):
    name : str = "calcular_emisiones"
    description : str = "Calcula las emisiones estimadas usando datos del producto"

    def _run(self, info_producto: str):
        # Aquí haces el parseo del texto y haces cálculos con pandas
        # Ejemplo: parsear que hay 2kg de acero -> buscar en CSV cuánto CO₂ por kg
        return "Emisión estimada: 3.4 kg CO₂e"
    



In [73]:
created_tools = [
    LeerPDFTool(),
    ConsultarCSVTool(),
    CalculadoraEmisionesTool()
]

In [74]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite", google_api_key=API_KEY, temperature=0)
tools = load_tools(['llm-math'], llm=llm)


agent = initialize_agent(
    tools + created_tools,
    llm,
    agent_type="zero-shot-react-description",
    memory=ConversationBufferMemory(memory_key="chat_history"),
    verbose=True
)

In [75]:
FILE = 'temp-files/temp.pdf'
CSV = 'data/processed/openlca_results_recorrected_calcsetup_ELCD.csv'

In [76]:
agent_prompt = f"""
Eres un analista de productos especializado en calcular la huella de carbono de un producto, tu objetivo es utilizar los datos de un archivo pdf de un producto y calcular cual es su huella estimada.
Para esto consultaras el CSV que tienes como herramienta donde tienes una lista de procesos y su emision de carbono. Compararas los procesos del producto y estimaras la emision de cada proceso.
Al final calcualras el total de emision de CO2 del producto
Esta bien que hagas asunciones si ves que te faltan datos, intenta que estas asunciones sean lo mas precisas posibles
PDF: {FILE}
CSV: {CSV}
"""

In [77]:
respuesta = agent.run(agent_prompt)
print(respuesta)



> Entering new AgentExecutor chain...
I need to calculate the carbon footprint of a product using a PDF and a CSV file. I will first read the PDF to understand the product's processes. Then, I will use the CSV to find emission factors for those processes. Finally, I will calculate the total emissions. I will make assumptions if necessary.

Action: leer_pdf
Action Input: temp-files/temp.pdf
Observation: 🍞 PAN DE MOLDE BIMBO (Rebanada estándar de 
500g, 20 rebanadas) 
Materias Primas:​
Harina de trigo, agua, levadura, azúcar, sal y grasas vegetales. La mayor parte de los 
ingredientes se obtienen de proveedores nacionales en cada país de producción. 
Fabricación:​
Mezclado, amasado, fermentación, horneado y enfriado en línea automatizada. Hornos de 
gas natural o eléctricos según la planta. 
Packaging:​
Bolsa plástica de polietileno de baja densidad (LDPE) con cierre. Algunas plantas utilizan 
bolsas con contenido reciclado. 
Transporte:​
Distribución diaria mediante furgonetas propias

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 1000000
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 31
}
].


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash-lite"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 1000000
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 28
}
]